In [ ]:
## TASCA 1. Presentació del conjunt de dades

El conjunt de dades correspon a una clínica veterinària que recull informació sobre els serveis d'atenció a animals en diferents centres de Catalunya. 
L'objectiu de l'anàlisi és estudiar l'activitat veterinària, comparant els serveis, les espècies ateses, els centres, el nombre d'animals i la durada de les visites. 
També es vol detectar possibles errors o dades incompletes abans de fer l'anàlisi.

El fitxer conté 120 registres i les columnes següents:

- **servei**: tipus de servei veterinari realitzat(consulta, vacuna, radiografia, etc)
- **espècie**: espècie de l'animal atès(gos, gat, conill, ocells, exòtics...)
- **centre**: municipi o centre veterinari on s'ha realitzat l'atenció.
- **animals_atesos**: nombre d'animals atesos en el servei.
- **durada_min**: durada del servei en minuts.
- **data**: data en què es va realitzar l'atenció.

Les dades presenten alguns problemes de qualitat, com ara camps buits, espais sobrants, valors no numèrics i dates incorrectes. 
Per tant, abans de calcular estadístiques o crear gràfics, serà necessari netejar i validar el conjunt de dades.

In [ ]:
## Fases de l'anàlisi

1.	**Presentació del conjunt de dades:** descripció del sector, objectiu de l’anàlisi i explicació de les columnes.
2.	**Exploració inicial:** càrrega del CSV, visualització de les primeres files, dimensions del conjunt de dades i tipus de dades.
3.	**Detecció d'incidències:** identificació de camps obligatoris buits, valors numèrics incorrectes, dates invàlides i inconsistències de format.
4.	**Neteja i validació:** aplicació de les transformacions necessàries i definició de les regles per determinar quins registres són vàlids.
5.	**Gestió dels registres incorrectes:** identificació dels registres rebutjats i explicació dels motius pels quals no es poden acceptar.
6.	**Anàlisi de les dades netes:** càlcul d’estadístiques i indicadors rellevants, amb agrupacions per les variables de classificació quan sigui convenient.
7.	**Resultats i conclusions:** resum dels principals resultats i valoració de la qualitat de les dades


In [1]:
# EXERCICI 1 - Exploració inicial

import csv
from datetime import datetime

nom_fitxer = "clinica_veterinaria.csv"

with open(nom_fitxer, mode="r", encoding="utf-8-sig", newline="") as fitxer: #-sig elimina el BOM
    lector = csv.DictReader(fitxer)
    camps = lector.fieldnames
    registres = list(lector)



print("Camps del CSV:", camps)
print("Nombre total de registres:", len(registres))
if registres:
    print("Exemple de primer registre:", registres[0])

Camps del CSV: ['servei', 'especie', 'centre', 'animals_atesos', 'durada_min', 'data']
Nombre total de registres: 120
Exemple de primer registre: {'servei': 'Microxip', 'especie': 'Gos', 'centre': 'Girona', 'animals_atesos': '12', 'durada_min': '114,0', 'data': '2025-04-21'}


In [ ]:
## EXERCICI 2 - Detecció de problemes
import csv
from datetime import datetime

nom_fitxer = "clinica_veterinaria.csv"

with open(nom_fitxer, mode="r", encoding="utf-8-sig", newline="") as fitxer:
    lector = csv.DictReader(fitxer)
    registres = list(lector)

comptador_problemes = {
    "camps_buit": 0,
    "animals_invalids": 0,
    "durada_invalides": 0,
    "dates_invalides": 0
}

def data_valida(valor):
    try:
        datetime.strptime(str(valor).strip(), "%Y-%m-%d")
        return True
    except (ValueError, TypeError):
        return False

for idx, reg in enumerate(registres, start=2):

    # 1. Buits en obligatoris
    for camp in ["servei", "especie", "centre"]:
        if not str(reg.get(camp, "")).strip():
            comptador_problemes["camps_buit"] += 1

    # 2. Animals atesos (enter > 0)
    try:
        val = int(str(reg.get("animals_atesos", "")).strip())
        if val <= 0:
            comptador_problemes["animals_invalids"] += 1
    except ValueError:
        comptador_problemes["animals_invalids"] += 1

    # 3. Durada (float > 0, acceptant coma)
    try:
        v_str = str(reg.get("durada_min", "")).strip().replace(",", ".")
        val_f = float(v_str)
        if val_f <= 0:
            comptador_problemes["durada_invalides"] += 1
    except ValueError:
        comptador_problemes["durada_invalides"] += 1

    # 4. Format de data
    if not data_valida(reg.get("data", "")):
        comptador_problemes["dates_invalides"] += 1

print("Resum d'incidències detectades:", comptador_problemes)

Resum d'incidències detectades: 121 {'camps_buit': 14, 'animals_invalids': 12, 'durada_invalides': 12, 'dates_invalides': 11}


In [ ]:
# EXERCICI 3 - Neteja i validació:

from datetime import datetime

#Funcions de conversió
def converteix_enter_positiu(valor):
    """Converteix a enter positiu (>0) o retorna None si no és vàlid."""
    try:
        v = int(str(valor).strip())
        return v if v > 0 else None
    except (ValueError, TypeError):
        return None

def converteix_durada_decimal(valor):
    """Accepta coma o punt decimal; retorna float positiu (>0) o None."""
    try:
        v_str = str(valor).strip().replace(",", ".")
        v = float(v_str)
        return v if v > 0 else None
    except (ValueError, TypeError):
        return None

def es_data_valida(valor: str) -> bool:
    """Valida el format AAAA-MM-DD i l'existència real de la data."""
    try:
        datetime.strptime(str(valor).strip(), "%Y-%m-%d")
        return True
    except (ValueError, TypeError):
        return False


serveis_nets = []
incidencies_neteja = []
registres_rebutjats = []

camps_obligatoris = ["servei", "especie", "centre"]

#Neteja
for idx, reg in enumerate(registres, start=2): # Fila 2 tenint en compte la capçalera
    es_valid = True
    reg_net = {}
    
    #1. Validació de camps obligatoris de text
    for camp in camps_obligatoris:
        val_orig = reg.get(camp, "")
        val_net = str(val_orig).strip()
        if not val_net:
            es_valid = False
            incidencies_neteja.append({
                "fila": idx, "tipus": "REBUTJAT", "camp": camp,
                "valor_original": val_orig, "motiu": "Camp obligatori buit"
            })
        else:
            if val_orig != val_net:
                incidencies_neteja.append({
                    "fila": idx, "tipus": "CORRECCIO", "camp": camp,
                    "valor_original": val_orig, "motiu": "Espais sobrants eliminats"
                })
            reg_net[camp] = val_net
            
    #2. Validació i conversió d'animals atesos
    anim_orig = reg.get("animals_atesos")
    anim_val = converteix_enter_positiu(anim_orig)
    if anim_val is None:
        es_valid = False
        incidencies_neteja.append({
            "fila": idx, "tipus": "REBUTJAT", "camp": "animals_atesos",
            "valor_original": anim_orig, "motiu": "Enter no positiu o invàlid"
        })
    else:
        reg_net["animals_atesos"] = anim_val
        
    #3. Validació i conversió de la durada en minuts (gestió de comes/punts)
    durada_orig = str(reg.get("durada_min", ""))
    durada_val = converteix_durada_decimal(durada_orig)
    if durada_val is None:
        es_valid = False
        incidencies_neteja.append({
            "fila": idx, "tipus": "REBUTJAT", "camp": "durada_min",
            "valor_original": durada_orig, "motiu": "Durada no numèrica o <= 0"
        })
    else:
        if "," in durada_orig:
            incidencies_neteja.append({
                "fila": idx, "tipus": "CORRECCIO", "camp": "durada_min",
                "valor_original": durada_orig, "motiu": "Coma convertida a punt decimal"
            })
        reg_net["durada_min"] = durada_val
        
    #4. Validació de la data
    data_orig = str(reg.get("data", "")).strip()
    if not es_data_valida(data_orig):
        es_valid = False
        incidencies_neteja.append({
            "fila": idx, "tipus": "REBUTJAT", "camp": "data",
            "valor_original": data_orig, "motiu": "Data invàlida o format incorrecte"
        })
    else:
        reg_net["data"] = data_orig
        
    # Classificació final del registre
    if es_valid:
        # Afegim una mètrica calculada equivalent a l'import: el temps total d'assistència (animals * minuts)
        reg_net["temps_total_min"] = round(reg_net["animals_atesos"] * reg_net["durada_min"], 2)
        serveis_nets.append(reg_net)
    else:
        registres_rebutjats.append(reg)

#Resultats
print(f"Total registres inicials: {len(registres)}")
print(f"Serveis nets i vàlids acceptats: {len(serveis_nets)}")
print(f"Registres rebutjats: {len(registres_rebutjats)}")
print(f"Incidències totals registrades (correccions + errors): {len(incidencies_neteja)}")

Total registres inicials: 120
Serveis nets i vàlids acceptats: 90
Registres rebutjats: 30
Incidències totals registrades (correccions + errors): 65


In [ ]:
## EXERCICI 4 - Gestió de registres incorrectes

import csv


Fitxer 'serveis_nets.csv' generat correctament amb 90 registres.


In [ ]:
## EXERCICI 5 - Anàlisi de dades netes

import csv

In [ ]:
## EXECICI 6 - Resultats i Conclusions